#### Neural Machine Translation Demo with Marian (Assignment Submission)
#### Transformer Architecture

In [2]:
%pwd

'/mnt/data/personal_projects/class-22/NandarLinn-NMT-Assignment-7'

In [3]:
!nvidia-smi

Mon Jun  1 15:58:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060        Off |   00000000:06:00.0  On |                  N/A |
|  0%   34C    P8             14W /  170W |     413MiB /  12288MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#### Shell Script Preparation for Tranformer Architecture

In [4]:
from IPython.display import Markdown

# Read the file content
with open('./transformer.phmy.sh', 'r') as f:
    content = f.read()

# Display it as a highlighted Bash block
display(Markdown(f"```bash\n{content}\n```"))

```bash
#!/bin/bash

## Written by Ye Kyaw Thu, Affiliated Professor, CADT, Cambodia
## for NMT Experiments between Burmese and Ethnic Languages
## used Marian NMT Framework for training
## Last updated: 23 May 2022


model_folder="model.transformer.phmy";
mkdir ${model_folder};
data_path="/mnt/data/personal_projects/class-22/NMT-notebooks/g2p-par";
src="ph"; tgt="my";

~/marian/build/marian \
    --model ${model_folder}/model.npz --type transformer \
    --train-sets ${data_path}/train.${src} ${data_path}/train.${tgt} \
    --max-length 200 \
    --vocabs ${data_path}/vocab/vocab.${src}.yml ${data_path}/vocab/vocab.${tgt}.yml \
    --mini-batch-fit -w 1000 --maxi-batch 100 \
    --early-stopping 10 \
    --valid-freq 5000 --save-freq 5000 --disp-freq 500 \
    --valid-metrics cross-entropy perplexity bleu \
    --valid-sets ${data_path}/dev.${src} ${data_path}/dev.${tgt} \
    --valid-translation-output ${model_folder}/valid.${src}-${tgt}.output --quiet-translation \
    --valid-mini-batch 64 \
    --beam-size 6 --normalize 0.6 \
    --log ${model_folder}/train.log --valid-log ${model_folder}/valid.log \
    --enc-depth 2 --dec-depth 2 \
    --transformer-heads 8 \
    --transformer-postprocess-emb d \
    --transformer-postprocess dan \
    --transformer-dropout 0.3 --label-smoothing 0.1 \
    --learn-rate 0.0003 --lr-warmup 0 --lr-decay-inv-sqrt 16000 --lr-report \
    --clip-norm 5 \
    --tied-embeddings \
    --devices 0 --sync-sgd --seed 1111 \
    --exponential-smoothing \
    --dump-config > ${model_folder}/${src}-${tgt}.config.yml
    
time ~/marian/build/marian -c ${model_folder}/${src}-${tgt}.config.yml  2>&1 | tee ${model_folder}/transformer-${src}-${tgt}.log
```

#### Training Transformer Model for Phoneme to Grapheme Translation

In [5]:
%pwd

'/mnt/data/personal_projects/class-22/NandarLinn-NMT-Assignment-7'

In [6]:
!./transformer.phmy.sh

[2026-05-26 22:59:41] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-05-26 22:59:41] [marian] Running on lst-hpc3090 as process 1364667 with command line:
[2026-05-26 22:59:41] [marian] marian -c model.transformer.phmy/ph-my.config.yml
[2026-05-26 22:59:41] [config] after: 0e
[2026-05-26 22:59:41] [config] after-batches: 0
[2026-05-26 22:59:41] [config] after-epochs: 0
[2026-05-26 22:59:41] [config] all-caps-every: 0
[2026-05-26 22:59:41] [config] allow-unk: false
[2026-05-26 22:59:41] [config] authors: false
[2026-05-26 22:59:41] [config] beam-size: 6
[2026-05-26 22:59:41] [config] bert-class-symbol: "[CLS]"
[2026-05-26 22:59:41] [config] bert-mask-symbol: "[MASK]"
[2026-05-26 22:59:41] [config] bert-masking-fraction: 0.15
[2026-05-26 22:59:41] [config] bert-sep-symbol: "[SEP]"
[2026-05-26 22:59:41] [config] bert-train-type-embeddings: true
[2026-05-26 22:59:41] [config] bert-type-vocab-size: 2
[2026-05-26 22:59:41] [config] build-info: ""
[2026-05-26 22:59:41] [conf

#### Checking Models  

In [6]:
!ls ./model.transformer.phmy

model.iter10000.npz  model.iter45000.npz      model.npz.progress.yml
model.iter15000.npz  model.iter50000.npz      model.npz.yml
model.iter20000.npz  model.iter5000.npz       ph-my.config.yml
model.iter25000.npz  model.iter55000.npz      train.log
model.iter30000.npz  model.npz		      transformer-ph-my.log
model.iter35000.npz  model.npz.decoder.yml    valid.log
model.iter40000.npz  model.npz.optimizer.npz  valid.ph-my.output


In [7]:
!cat ./model.transformer.phmy/valid.log

[2026-05-31 23:13:44] [valid] Ep. 80 : Up. 5000 : cross-entropy : 1.81503 : new best
[2026-05-31 23:13:44] [valid] Ep. 80 : Up. 5000 : perplexity : 1.60357 : new best
[2026-05-31 23:13:44] [valid] First sentence's tokens as scored:
[2026-05-31 23:13:44] [valid] DefaultVocab keeps original segments for scoring
[2026-05-31 23:13:44] [valid]   Hyp: ဥတ် တ ရ ဖ လ ဂု နီ
[2026-05-31 23:13:44] [valid]   Ref: ဥတ် တ ရ ဖ လ ဂု နီ
[2026-05-31 23:13:44] [valid] Ep. 80 : Up. 5000 : bleu : 75.9618 : new best
[2026-05-31 23:16:26] [valid] Ep. 159 : Up. 10000 : cross-entropy : 2.0176 : stalled 1 times (last best: 1.81503)
[2026-05-31 23:16:26] [valid] Ep. 159 : Up. 10000 : perplexity : 1.69036 : stalled 1 times (last best: 1.60357)
[2026-05-31 23:16:26] [valid] Ep. 159 : Up. 10000 : bleu : 76.0964 : new best
[2026-05-31 23:19:07] [valid] Ep. 239 : Up. 15000 : cross-entropy : 2.16604 : stalled 2 times (last best: 1.81503)
[2026-05-31 23:19:07] [valid] Ep. 239 : Up. 15000 : perplexity : 1.75692 : stalled 2

#### Testing Transformer Model for Phoneme to Grapheme Translation

In [8]:
!~/marian/build/marian-decoder --help

Marian: Fast Neural Machine Translation in C++
Usage: /home/elio/marian/build/marian-decoder [OPTIONS]

General options:
  -h,--help                             Print this help message and exit
  --version                             Print the version number and exit
  --authors                             Print list of authors and exit
  --cite                                Print citation and exit
  --build-info TEXT                     Print CMake build options and exit. Set to 'all' to print advanced options
  -c,--config VECTOR ...                Configuration file(s). If multiple, later overrides earlier
  -w,--workspace INT=512                Preallocate arg MB of work space. Negative `--workspace -N` value allocates workspace as total available GPU memory minus N megabytes.
  --log TEXT                            Log training process information to file given by arg
  --log-level TEXT=info                 Set verbosity level of logging: trace, debug, info, warn, err(or), critic

In [10]:
!time ~/marian/build/marian-decoder -m ./model.transformer.phmy/model.npz -v ./g2p-par/vocab/vocab.ph.yml ./g2p-par/vocab/vocab.my.yml --devices 0 < ./g2p-par/test.ph > ./transformer.phmy.hyp.txt

[2026-06-01 15:59:56] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-06-01 15:59:56] [marian] Running on bumblebee as process 22028 with command line:
[2026-06-01 15:59:56] [marian] /home/elio/marian/build/marian-decoder -m ./model.transformer.phmy/model.npz -v ./g2p-par/vocab/vocab.ph.yml ./g2p-par/vocab/vocab.my.yml --devices 0
[2026-06-01 15:59:56] [config] alignment: ""
[2026-06-01 15:59:56] [config] allow-special: false
[2026-06-01 15:59:56] [config] allow-unk: false
[2026-06-01 15:59:56] [config] authors: false
[2026-06-01 15:59:56] [config] beam-size: 12
[2026-06-01 15:59:56] [config] bert-class-symbol: "[CLS]"
[2026-06-01 15:59:56] [config] bert-mask-symbol: "[MASK]"
[2026-06-01 15:59:56] [config] bert-masking-fraction: 0.15
[2026-06-01 15:59:56] [config] bert-sep-symbol: "[SEP]"
[2026-06-01 15:59:56] [config] bert-train-type-embeddings: true
[2026-06-01 15:59:56] [config] bert-type-vocab-size: 2
[2026-06-01 15:59:56] [config] best-deep: false
[2026-06-01 15:5

**Marian က training/testing အတွက် အရမ်းမြန်ပါတယ်။ အဲဒါကြောင့် ဆရာ ကြိုက်တယ်။**

## Evaluation

In [11]:
!perl /home/elio/tool/mosesbin/ubuntu-17.04/moses/scripts/generic/multi-bleu.perl ./g2p-par/test.my < ./transformer.phmy.hyp.txt

BLEU = 77.52, 87.7/79.0/74.2/70.3 (BP=1.000, ratio=1.000, hyp_len=8044, ref_len=8047)
